This notebook is designed to be run in sequential order. The wandb report can be found below.
https://api.wandb.ai/links/pkotchav-carnegie-mellon-university/m1xnkkuk


The final model used was the ASR Network as is in this current version of the notebook.

# Installs

In [ ]:
%pip install torch==1.13.1+cu117 torchvision==0.14.1+cu117 torchtext==0.14.1 torchaudio==0.13.1 torchdata==0.5.1 --extra-index-url https://download.pytorch.org/whl/cu117 -q


This may take a while

In [ ]:
!pip install wandb --quiet
!pip install python-Levenshtein -q
!pip install torchsummaryX==1.3.0
!pip install pandas==1.5.2
# !git clone --recursive https://github.com/parlance/ctcdecode.git
# !pip install wget -q
# %cd ctcdecode
# !pip install . -q
# %cd ..

# Imports

In [ ]:
import torch
import random
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torchsummaryX import summary
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

import torchaudio.transforms as tat

from sklearn.metrics import accuracy_score
import gc

import zipfile
import pandas as pd
from tqdm import tqdm
import os
import datetime

# imports for decoding and distance calculation
import ctcdecode
import Levenshtein
from ctcdecode import CTCBeamDecoder

import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device: ", device)

Device:  cuda


In [ ]:
print(pd.__version__)

1.5.2


In [ ]:
pip show torchsummaryX

Name: torchsummaryX
Version: 1.3.0
Summary: Improved visualization tool of torchsummary.
Home-page: https://github.com/nmhkahn/torchsummaryX
Author: Namhyuk Ahn
Author-email: nmhkahn@gmail.com
License: UNKNOWN
Location: /usr/local/lib/python3.10/dist-packages
Requires: numpy, pandas, torch
Required-by: 


# Kaggle Setup

In [ ]:
!pip install --upgrade --force-reinstall --no-deps kaggle==1.5.8 -q
!mkdir /root/.kaggle

with open("/root/.kaggle/kaggle.json", "w+") as f:
    f.write('{"username":"paulkotchavong","key":"42477bb8c1877b51a852f26c36c06fd3"}')

!chmod 600 /root/.kaggle/kaggle.json

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
!kaggle competitions download -c hw3p2-785-f24

100% 3.96G/3.97G [00:42<00:00, 142MB/s]
100% 3.97G/3.97G [00:43<00:00, 99.2MB/s]


In [ ]:
'''
This will take a couple minutes, but you should see at least the following:
11-785-f24-hw3p2  ctcdecode  hw3p2asr-f24.zip  sample_data
'''
!unzip -q hw3p2-785-f24.zip
!ls

11785-f24-hw3p2  ctcdecode  hw3p2-785-f24.zip  sample_data


# Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

# Dataset and Dataloader

In [ ]:
# ARPABET PHONEME MAPPING
# DO NOT CHANGE

CMUdict_ARPAbet = {
    "" : " ",
    "[SIL]": "-", "NG": "G", "F" : "f", "M" : "m", "AE": "@",
    "R"    : "r", "UW": "u", "N" : "n", "IY": "i", "AW": "W",
    "V"    : "v", "UH": "U", "OW": "o", "AA": "a", "ER": "R",
    "HH"   : "h", "Z" : "z", "K" : "k", "CH": "C", "W" : "w",
    "EY"   : "e", "ZH": "Z", "T" : "t", "EH": "E", "Y" : "y",
    "AH"   : "A", "B" : "b", "P" : "p", "TH": "T", "DH": "D",
    "AO"   : "c", "G" : "g", "L" : "l", "JH": "j", "OY": "O",
    "SH"   : "S", "D" : "d", "AY": "Y", "S" : "s", "IH": "I",
    "[SOS]": "[SOS]", "[EOS]": "[EOS]"
}

CMUdict = list(CMUdict_ARPAbet.keys())
ARPAbet = list(CMUdict_ARPAbet.values())


PHONEMES = CMUdict[:-2]
LABELS = ARPAbet[:-2]



In [ ]:
# put the phonemes into alphabetical order
PHONEMES.sort()
LABELS = [CMUdict_ARPAbet[i] for i in PHONEMES]

In [ ]:
# You might want to play around with the mapping as a sanity check here
print(PHONEMES)
print(LABELS)
print(len(PHONEMES))
print(len(LABELS))

['', '[SIL]', 'NG', 'F', 'M', 'AE', 'R', 'UW', 'N', 'IY', 'AW', 'V', 'UH', 'OW', 'AA', 'ER', 'HH', 'Z', 'K', 'CH', 'W', 'EY', 'ZH', 'T', 'EH', 'Y', 'AH', 'B', 'P', 'TH', 'DH', 'AO', 'G', 'L', 'JH', 'OY', 'SH', 'D', 'AY', 'S', 'IH']
['_', '-', 'G', 'f', 'm', '@', 'r', 'u', 'n', 'i', 'W', 'v', 'U', 'o', 'a', 'R', 'h', 'z', 'k', 'C', 'w', 'e', 'Z', 't', 'E', 'y', 'A', 'b', 'p', 'T', 'D', 'c', 'g', 'l', 'j', 'O', 'S', 'd', 'Y', 's', 'I']
41
41


In [ ]:
mfcc_dir = os.path.join('/content/11785-f24-hw3p2', "train-clean-100", 'mfcc')
transcript_dir = os.path.join('/content/11785-f24-hw3p2', "train-clean-100", 'transcript')

# might as well load it in order
# mfcc_name = os.listdir(self.mfcc_dir)
# mfcc_name.sort()
# transcript_name = os.listdir(self.transcript_dir)
# transcript_name.sort()
mfcc_files = sorted(os.listdir(mfcc_dir))
transcript_files = sorted(os.listdir(transcript_dir))
print(len(mfcc_files))
print(len(transcript_files))

features = []
labels = []

for i in range(len(mfcc_files)):
    mfcc = np.load(os.path.join(mfcc_dir, mfcc_files[i]))
    transcript = np.load(os.path.join(transcript_dir, transcript_files[i]))

    mfcc = (mfcc - np.mean(mfcc, axis=0, keepdims=True))/(np.std(mfcc, axis=0, keepdims=True) + 1e-5)
    transcript = transcript[1:-1]

    features.append(mfcc)
    labels.append(transcript)

28539
28539


KeyboardInterrupt: 

In [ ]:
for i in range(3):
    print(features[i].shape)
    print(labels[i].shape)
    print(features[i])
    print(labels[i])


(1404, 28)
(145,)
[[-0.18250355  0.4172915   0.3693044  ... -0.01691582  0.72973764
  -1.5239604 ]
 [-0.13833714  0.48505074  0.4269905  ... -0.06249875 -0.6334915
  -1.4798307 ]
 [-0.17921834  0.6812124   0.49413612 ... -0.29598892  0.80933577
  -1.4886994 ]
 ...
 [-0.7767339  -1.3129497   0.8417389  ... -1.2554015  -2.718656
  -0.13059846]
 [-0.9594661  -1.270711    0.70879424 ... -0.69407827 -1.2275912
  -0.06935827]
 [-1.0557525  -1.2458974   0.6029278  ... -0.7608054  -0.83664864
   0.00604726]]
['[SIL]' 'CH' 'AE' 'P' 'T' 'ER' 'W' 'AH' 'N' '[SIL]' 'M' 'IH' 'S' 'IH' 'Z'
 'R' 'EY' 'CH' 'AH' 'L' 'IH' 'N' 'D' 'IH' 'Z' 'S' 'ER' 'P' 'R' 'AY' 'Z'
 'D' '[SIL]' 'M' 'IH' 'S' 'IH' 'Z' 'R' 'EY' 'CH' 'AH' 'L' 'IH' 'N' 'D'
 '[SIL]' 'L' 'AY' 'V' 'D' '[SIL]' 'JH' 'AH' 'S' 'T' 'W' 'EH' 'R' 'DH' 'AH'
 'AE' 'V' 'AH' 'N' 'L' 'IY' 'AH' 'M' 'EY' 'N' 'R' 'OW' 'D' '[SIL]' 'D'
 'IH' 'P' 'T' '[SIL]' 'D' 'AW' 'N' 'IH' 'N' 'T' 'UW' 'AH' 'L' 'IH' 'T'
 'AH' 'L' 'HH' 'AA' 'L' 'OW' '[SIL]' 'F' 'R' 'IH' 'N' 'JH' 

In [ ]:
yep = np.concatenate(labels, axis=0)
print(yep.shape)

(3717031,)


In [ ]:
print(yep[146])

DH


In [ ]:
PHONEME_MAP = {}
for i in range(len(PHONEMES)):
    PHONEME_MAP[PHONEMES[i]] = i

print(PHONEME_MAP)

{'': 0, 'AA': 1, 'AE': 2, 'AH': 3, 'AO': 4, 'AW': 5, 'AY': 6, 'B': 7, 'CH': 8, 'D': 9, 'DH': 10, 'EH': 11, 'ER': 12, 'EY': 13, 'F': 14, 'G': 15, 'HH': 16, 'IH': 17, 'IY': 18, 'JH': 19, 'K': 20, 'L': 21, 'M': 22, 'N': 23, 'NG': 24, 'OW': 25, 'OY': 26, 'P': 27, 'R': 28, 'S': 29, 'SH': 30, 'T': 31, 'TH': 32, 'UH': 33, 'UW': 34, 'V': 35, 'W': 36, 'Y': 37, 'Z': 38, 'ZH': 39, '[SIL]': 40}


In [ ]:
labels_mapped = []
for i in range(len(labels)):
    labels_mapped.append([PHONEME_MAP[j] for j in labels[i]])
print(labels[0])
print(labels_mapped[0])

['[SIL]' 'CH' 'AE' 'P' 'T' 'ER' 'W' 'AH' 'N' '[SIL]' 'M' 'IH' 'S' 'IH' 'Z'
 'R' 'EY' 'CH' 'AH' 'L' 'IH' 'N' 'D' 'IH' 'Z' 'S' 'ER' 'P' 'R' 'AY' 'Z'
 'D' '[SIL]' 'M' 'IH' 'S' 'IH' 'Z' 'R' 'EY' 'CH' 'AH' 'L' 'IH' 'N' 'D'
 '[SIL]' 'L' 'AY' 'V' 'D' '[SIL]' 'JH' 'AH' 'S' 'T' 'W' 'EH' 'R' 'DH' 'AH'
 'AE' 'V' 'AH' 'N' 'L' 'IY' 'AH' 'M' 'EY' 'N' 'R' 'OW' 'D' '[SIL]' 'D'
 'IH' 'P' 'T' '[SIL]' 'D' 'AW' 'N' 'IH' 'N' 'T' 'UW' 'AH' 'L' 'IH' 'T'
 'AH' 'L' 'HH' 'AA' 'L' 'OW' '[SIL]' 'F' 'R' 'IH' 'N' 'JH' 'D' 'W' 'IH'
 'DH' 'AA' 'L' 'D' 'ER' 'Z' 'AH' 'N' 'D' 'L' 'EY' 'D' 'IY' 'Z' 'IH' 'R'
 'D' 'R' 'AA' 'P' 'S' 'AH' 'N' 'D' 'T' 'R' 'AE' 'V' 'ER' 'S' 'T' 'B' 'AY'
 'AH' 'B' 'R' 'UH' 'K' '[SIL]']
[40, 8, 2, 27, 31, 12, 36, 3, 23, 40, 22, 17, 29, 17, 38, 28, 13, 8, 3, 21, 17, 23, 9, 17, 38, 29, 12, 27, 28, 6, 38, 9, 40, 22, 17, 29, 17, 38, 28, 13, 8, 3, 21, 17, 23, 9, 40, 21, 6, 35, 9, 40, 19, 3, 29, 31, 36, 11, 28, 10, 3, 2, 35, 3, 23, 21, 18, 3, 22, 13, 23, 28, 25, 9, 40, 9, 17, 27, 31, 40, 9, 5, 23, 17, 

### Train Data

In [ ]:
class AudioDataset(torch.utils.data.Dataset):

    # For this homework, we give you full flexibility to design your data set class.
    # Hint: The data from HW1 is very similar to this HW

    #TODO
    def __init__(self, root, partition= "train-clean-100", file_subset=None):
        '''
        Initializes the dataset.

        INPUTS: What inputs do you need here?
        '''
        self.root = root
        self.partition = partition
        self.file_subset = file_subset
        # Load the directory and all files in them

        self.mfcc_dir = os.path.join(self.root, self.partition, 'mfcc')
        self.transcript_dir = os.path.join(self.root, self.partition, 'transcript')

        # might as well load it in order
        # mfcc_name = os.listdir(self.mfcc_dir)
        # mfcc_name.sort()
        # transcript_name = os.listdir(self.transcript_dir)
        # transcript_name.sort()
        self.mfcc_files = sorted(os.listdir(self.mfcc_dir))
        self.transcript_files = sorted(os.listdir(self.transcript_dir))

        self.PHONEMES = PHONEMES

        #TODO
        # WHAT SHOULD THE LENGTH OF THE DATASET BE?
        self.length = len(self.mfcc_files)
        assert len(self.mfcc_files) == len(self.transcript_files)

        #TODO
        # HOW CAN WE REPRESENT PHONEMES? CAN WE CREATE A MAPPING FOR THEM?
        # HINT: TENSORS CANNOT STORE NON-NUMERICAL VALUES OR STRINGS
        self.PHONEME_MAP = {}
        for i in range(len(self.PHONEMES)):
            self.PHONEME_MAP[self.PHONEMES[i]] = i


        #TODO
        # CREATE AN ARRAY OF ALL FEATUERS AND LABELS
        # WHAT NORMALIZATION TECHNIQUE DID YOU USE IN HW1? CAN WE USE IT HERE?
        self.mfccs = []
        self.transcripts = []

        for i in range(len(self.mfcc_files)):
            mfcc = np.load(os.path.join(self.mfcc_dir, self.mfcc_files[i]))
            transcript = np.load(os.path.join(self.transcript_dir, self.transcript_files[i]))

            mfcc = (mfcc - np.mean(mfcc, axis=0, keepdims=True))/(np.std(mfcc, axis=0, keepdims=True) + 1e-5)
            transcript = transcript[1:-1]

            self.mfccs.append(torch.tensor(mfcc, dtype=torch.float32))
            self.transcripts.append(transcript)

        self.transcripts_mapped = []
        for i in range(len(self.transcripts)):
            self.transcripts_mapped.append(torch.tensor([self.PHONEME_MAP[j] for j in self.transcripts[i]]))

        '''
        You may decide to do this in __getitem__ if you wish.
        However, doing this here will make the __init__ function take the load of
        loading the data, and shift it away from training.
        '''


    def __len__(self):

        '''
        TODO: What do we return here?
        '''
        return self.length

    def __getitem__(self, ind):
        '''
        TODO: RETURN THE MFCC COEFFICIENTS AND ITS CORRESPONDING LABELS

        If you didn't do the loading and processing of the data in __init__,
        do that here.

        Once done, return a tuple of features and labels.
        '''
        mfcc = self.mfccs[ind]
        transcript = self.transcripts_mapped[ind]
        return mfcc, transcript


    def collate_fn(self,batch):
        '''
        TODO:
        1.  Extract the features and labels from 'batch'
        2.  We will additionally need to pad both features and labels,
            look at pytorch's docs for pad_sequence
        3.  This is a good place to perform transforms, if you so wish.
            Performing them on batches will speed the process up a bit.
        4.  Return batch of features, labels, lenghts of features,
            and lengths of labels.
        '''
        # batch of input mfcc coefficients
        batch_mfcc = [item[0] for item in batch]
        # batch of output phonemes
        batch_transcript = [item[1] for item in batch]

        # HINT: CHECK OUT -> pad_sequence (imported above)
        # Also be sure to check the input format (batch_first)
        batch_mfcc_pad = pad_sequence(batch_mfcc, batch_first=True, padding_value=0.0)
        lengths_mfcc = [len(mfcc) for mfcc in batch_mfcc]

        batch_transcript_pad = pad_sequence(batch_transcript, batch_first=True, padding_value=-1.0)
        lengths_transcript = [len(transcript) for transcript in batch_transcript]

        # You may apply some transformation, Time and Frequency masking, here in the collate function;
        # Food for thought -> Why are we applying the transformation here and not in the __getitem__?
        #                  -> Would we apply transformation on the validation set as well?
        #                  -> Is the order of axes / dimensions as expected for the transform functions?
        if self.partition == 'train-clean-100':
            batch_mfcc_pad = self.apply_spec_augment(batch_mfcc_pad,time_mask_param=10, freq_mask_param=8)
        # Return the following values: padded features, padded labels, actual length of features, actual length of the labels
        return batch_mfcc_pad, batch_transcript_pad, torch.tensor(lengths_mfcc), torch.tensor(lengths_transcript)

    def apply_spec_augment(self, mfcc_batch, time_mask_param=10, freq_mask_param=8):
        """
        Applies SpecAugment to a batch of MFCCs.
        Arguments:
        - mfcc_batch: Tensor of shape (batch_size, time_steps, freq_bins)
        - time_mask_param: Max time steps to mask
        - freq_mask_param: Max frequency bins to mask
        """
        batch_size, time_steps, freq_bins = mfcc_batch.shape

        # Apply time masking
        for i in range(batch_size):
            t = random.randint(0, time_mask_param)
            t0 = random.randint(0, max(1, time_steps - t))
            mfcc_batch[i, t0:t0 + t, :] = 0

        # Apply frequency masking
        for i in range(batch_size):
            f = random.randint(0, freq_mask_param)
            f0 = random.randint(0, max(1, freq_bins - f))
            mfcc_batch[i, :, f0:f0 + f] = 0

        return mfcc_batch



### Test Data

In [ ]:
# Test Dataloader
#TODO
class AudioDatasetTest(torch.utils.data.Dataset):
    def __init__(self, root, partition= "test-clean"):
        '''
        Initializes the dataset.
        '''
        self.root = root
        self.partition = partition
        # Load the directory and all files in them

        self.mfcc_dir = os.path.join(self.root, self.partition, 'mfcc')
        self.mfcc_names = sorted(os.listdir(self.mfcc_dir))

        self.mfccs = []

        for i in range(len(self.mfcc_names)):
            mfcc = np.load(os.path.join(self.mfcc_dir, self.mfcc_names[i]))
            mfcc = (mfcc - np.mean(mfcc, axis=0, keepdims=True))/(np.std(mfcc, axis=0, keepdims=True) + 1e-5)
            print(mfcc.shape)
            self.mfccs.append(torch.tensor(mfcc, dtype=torch.float32))

        self.length = len(self.mfccs)

    def __len__(self):
        '''
        TODO: What do we return here?
        '''
        return self.length

    def __getitem__(self, ind):
        '''
        TODO: RETURN THE MFCC COEFFICIENTS
        '''
        mfcc = self.mfccs[ind]
        return mfcc

    def collate_fn(self,batch):
        '''
        TODO:
        1.  Extract the features and labels from 'batch'
        2.  We will additionally need to pad both features and labels,
            look at pytorch's docs for pad_sequence
        3.  This is a good place to perform transforms, if you so wish.
            Performing them on batches will speed the process up a bit.
        4.  Return batch of features, labels, lenghts of features,
            and lengths of labels.
        '''
        # batch of input mfcc coefficients
        batch_mfcc = [item for item in batch]

        # HINT: CHECK OUT -> pad_sequence (imported above)
        # Also be sure to check the input format (batch_first)
        batch_mfcc_pad = pad_sequence(batch_mfcc, batch_first=True, padding_value=0.0)
        lengths_mfcc = [len(mfcc) for mfcc in batch_mfcc]

        # You may apply some transformation, Time and Frequency masking, here in the collate function;
        # Food for thought -> Why are we applying the transformation here and not in the __getitem__?
        #                  -> Would we apply transformation on the validation set as well?
        #                  -> Is the order of axes / dimensions as expected for the transform functions?

        # Return the following values: padded features, padded labels, actual length of features, actual length of the labels
        return batch_mfcc_pad, torch.tensor(lengths_mfcc)



### Config - Hyperparameters

In [ ]:
root = '/content/11-785-f24-hw3p2/'

# Feel free to add more items here
config = {
    "beam_width" : 3,
    "lr"         : 2e-3,
    "epochs"     : 60,
    "batch_size" : 64  # Increase if your device can handle it
}

# You may pass this as a parameter to the dataset class above
# This will help modularize your implementation
transforms = [] # set of tranformations

### Data loaders

In [ ]:
# get me RAMMM!!!!
import gc
gc.collect()

0

In [ ]:
# Create objects for the dataset class
train_data = AudioDataset(root='/content/11785-f24-hw3p2', partition='train-clean-100')
val_data = AudioDataset(root='/content/11785-f24-hw3p2', partition='dev-clean')
test_data = AudioDatasetTest(root='/content/11785-f24-hw3p2', partition='test-clean')

# import random
# from torch.utils.data import Subset

# train_subset_size = int(1* len(train_data))
# val_subset_size = int(1 * len(val_data))

# train_subset_indices = random.sample(range(len(train_data)), train_subset_size)
# val_subset_indices = random.sample(range(len(val_data)), val_subset_size)

# train_data_sub = Subset(train_data, train_subset_indices)
# val_data_sub = Subset(val_data, val_subset_indices)

# Do NOT forget to pass in the collate function as parameter while creating the dataloader
train_loader = torch.utils.data.DataLoader(
    dataset=train_data,
    num_workers=4,
    batch_size=config['batch_size'],
    shuffle=True,
    collate_fn=train_data.collate_fn)
val_loader = torch.utils.data.DataLoader(
    dataset=val_data,
    num_workers=4,
    batch_size=config['batch_size'],
    shuffle=False,
    collate_fn=val_data.collate_fn)
test_loader = torch.utils.data.DataLoader(
    dataset=test_data,
    num_workers=4,
    batch_size=config['batch_size'],
    shuffle=False,
    collate_fn=test_data.collate_fn)

print("Batch size: ", config['batch_size'])
print("Train dataset samples = {}, batches = {}".format(train_data.__len__(), len(train_loader)))
print("Val dataset samples = {}, batches = {}".format(val_data.__len__(), len(val_loader)))
print("Test dataset samples = {}, batches = {}".format(test_data.__len__(), len(test_loader)))

(1039, 28)
(323, 28)
(658, 28)
(264, 28)
(517, 28)
(959, 28)
(1051, 28)
(423, 28)
(669, 28)
(1053, 28)
(436, 28)
(1240, 28)
(1160, 28)
(787, 28)
(218, 28)
(577, 28)
(350, 28)
(883, 28)
(1568, 28)
(1385, 28)
(1675, 28)
(651, 28)
(1113, 28)
(1323, 28)
(1161, 28)
(657, 28)
(397, 28)
(267, 28)
(779, 28)
(463, 28)
(267, 28)
(657, 28)
(405, 28)
(329, 28)
(577, 28)
(340, 28)
(321, 28)
(517, 28)
(204, 28)
(537, 28)
(1156, 28)
(213, 28)
(513, 28)
(532, 28)
(585, 28)
(340, 28)
(1494, 28)
(2001, 28)
(315, 28)
(1997, 28)
(1499, 28)
(1629, 28)
(471, 28)
(335, 28)
(902, 28)
(1165, 28)
(305, 28)
(311, 28)
(395, 28)
(1333, 28)
(559, 28)
(769, 28)
(217, 28)
(796, 28)
(1068, 28)
(900, 28)
(1792, 28)
(1257, 28)
(1061, 28)
(852, 28)
(236, 28)
(1420, 28)
(2071, 28)
(2302, 28)
(605, 28)
(1515, 28)
(1461, 28)
(298, 28)
(435, 28)
(1604, 28)
(1655, 28)
(457, 28)
(1150, 28)
(1389, 28)
(1022, 28)
(1398, 28)
(959, 28)
(2363, 28)
(1520, 28)
(741, 28)
(2008, 28)
(1120, 28)
(1896, 28)
(366, 28)
(187, 28)
(421, 28)
(

In [ ]:
print("Train dataset samples = {}, batches = {}".format(train_data_sub.__len__(), len(train_loader)))

Train dataset samples = 2853, batches = 45


In [ ]:
for i, data in enumerate(test_loader):
    x, lx = data
    print(x.shape, lx.shape)
    print(lx)
    break

torch.Size([64, 2001, 28]) torch.Size([64])
tensor([1039,  323,  658,  264,  517,  959, 1051,  423,  669, 1053,  436, 1240,
        1160,  787,  218,  577,  350,  883, 1568, 1385, 1675,  651, 1113, 1323,
        1161,  657,  397,  267,  779,  463,  267,  657,  405,  329,  577,  340,
         321,  517,  204,  537, 1156,  213,  513,  532,  585,  340, 1494, 2001,
         315, 1997, 1499, 1629,  471,  335,  902, 1165,  305,  311,  395, 1333,
         559,  769,  217,  796])


In [ ]:
# sanity check
for data in train_loader:
    x, y, lx, ly = data
    print(x.shape, y.shape, lx.shape, ly.shape)
    print(lx, ly)
    print(max(lx), max(ly))
    print(y[0])
    break

torch.Size([64, 1658, 28]) torch.Size([64, 203]) torch.Size([64]) torch.Size([64])
tensor([1439, 1379, 1568,  468, 1658, 1282, 1410, 1430, 1444, 1525, 1385, 1404,
        1271, 1169, 1516, 1434, 1390, 1358, 1261, 1322, 1522, 1538, 1499, 1622,
         829, 1056, 1250, 1348,  815, 1461,  404, 1651, 1560, 1351,  643, 1595,
        1365, 1073, 1528, 1591,  459,  226, 1559, 1420, 1448,  544, 1546, 1460,
        1230, 1637, 1369, 1610, 1589, 1034, 1510, 1609, 1385, 1512,  472, 1541,
         526, 1585, 1102, 1585]) tensor([132, 131, 169,  53, 203, 137, 134, 154, 141, 188, 105, 158,  82, 101,
        154, 127, 170, 137, 159, 130, 143, 168, 137, 155,  90, 117, 118, 174,
         72, 130,  45, 166, 166, 143,  75, 149, 146, 150, 162, 148,  48,  18,
        176, 106, 162,  62, 171, 146, 119, 146, 128, 168, 176, 110, 143, 129,
        137, 129,  62, 147,  56, 195, 130, 178])
tensor(1658) tensor(203)
tensor([ 1, 26,  8, 37, 39,  7,  8, 37, 38,  8, 40,  2,  5, 23, 33, 31,  2, 27,
        12,  8,  1

# NETWORK

## Basic

This is a basic block for understanding, you can skip this and move to pBLSTM one

In [ ]:
torch.cuda.empty_cache()

class Network(nn.Module):

    def __init__(self):

        super(Network, self).__init__()

        # Adding some sort of embedding layer or feature extractor might help performance.
        # Adding 1D CNN layer
        self.embedding = torch.nn.Sequential(
          nn.Conv1d(in_channels = 28, out_channels = 128, kernel_size = 3, padding=1,stride=1),
          nn.ReLU(),
          nn.BatchNorm1d(128),
        )

        # TODO : look up the documentation. You might need to pass some additional parameters.
        self.lstm = nn.LSTM(input_size = 128, hidden_size = 256, num_layers = 3, bidirectional=True)

        self.classification = nn.Sequential(
            #TODO: Linear layer with in_features from the lstm module above and out_features = OUT_SIZE
            nn.Linear(512, 41)
        )


        self.logSoftmax = nn.LogSoftmax(dim=1)#TODO: Apply a log softmax here. Which dimension would apply it on ?

    def forward(self, x, lx):
        x = x.transpose(1, 2)  # Transpose to (batch_size, input_size, sequence_length) for Conv1d
        x = self.embedding(x)  # Shape: (batch_size, out_channels, new_length)

        x = x.transpose(1, 2)  # Transpose back to (batch_size, sequence_length, out_channels) for LSTM
        x = pack_padded_sequence(x, lx, batch_first=True, enforce_sorted=False)

        # LSTM forward pass
        x, _ = self.lstm(x)

        # Unpack sequence
        x, lx = pad_packed_sequence(x, batch_first=True)

        x = self.classification(x)  # Shape: (batch_size, sequence_length, out_features)
        x = self.logSoftmax(x)       # Shape: (batch_size, sequence_length, out_features)

        return x, lx

## Initialize Basic Network
(If trying out the basic Network)

This network was not used as the final network

In [ ]:
torch.cuda.empty_cache()

model = Network().to(device)
summary(model, x.to(device), lx) # x and lx come from the sanity check above :)

                           Kernel Shape     Output Shape     Params  \
Layer                                                                 
0_embedding.Conv1d_0       [28, 128, 3]  [64, 128, 2936]     10.88k   
1_embedding.ReLU_1                    -  [64, 128, 2936]          -   
2_embedding.BatchNorm1d_2         [128]  [64, 128, 2936]      256.0   
3_lstm                                -     [42152, 512]  3.944448M   
4_classification.Linear_0     [512, 41]   [64, 2936, 41]    21.033k   
5_logSoftmax                          -   [64, 2936, 41]          -   

                            Mult-Adds  
Layer                                  
0_embedding.Conv1d_0       31.567872M  
1_embedding.ReLU_1                  -  
2_embedding.BatchNorm1d_2       128.0  
3_lstm                       3.93216M  
4_classification.Linear_0     20.992k  
5_logSoftmax                        -  
-------------------------------------------------------------------------------
                          Total

,Kernel Shape,Output Shape,Params,Mult-Adds
Layer,,,,
0_embedding.Conv1d_0,"[28, 128, 3]","[64, 128, 2936]",10880.0,31567872.0
1_embedding.ReLU_1,-,"[64, 128, 2936]",NaN,NaN
2_embedding.BatchNorm1d_2,[128],"[64, 128, 2936]",256.0,128.0
3_lstm,-,"[42152, 512]",3944448.0,3932160.0
4_classification.Linear_0,"[512, 41]","[64, 2936, 41]",21033.0,20992.0
5_logSoftmax,-,"[64, 2936, 41]",NaN,NaN


## ASR Network

Below is the final network that was used.

### Pyramid Bi-LSTM (pBLSTM)

In [ ]:
# Utils for network
torch.cuda.empty_cache()

class PermuteBlock(torch.nn.Module):
    def forward(self, x):
        return x.transpose(1, 2)

In [ ]:
class BiLSTM(torch.nn.Module):

    def __init__(self, input_size, hidden_size, num_layers=1):
        super(BiLSTM, self).__init__()

        self.blstm = torch.nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, bidirectional=True)
        self._init_weights()

    def _init_weights(self):
        for name, param in self.blstm.named_parameters():
            if 'weight_ih' in name:
                torch.nn.init.xavier_uniform_(param.data)
            elif 'weight_hh' in name:
                torch.nn.init.orthogonal_(param.data)
            elif 'bias' in name:
                param.data.fill_(0)

    def forward(self, x):
        x, _ = self.blstm(x)
        return x

In [ ]:
import torch
import torch.nn as nn
from torch.autograd import Variable

class LockedDropout(nn.Module):
    def __init__(self, dropout=0.5):
        super().__init__()
        self.dropout = dropout

    def forward(self, x):
        if not self.training:
            return x
        m = x.data.new(1, x.size(1), x.size(2)).bernoulli_(1 - self.dropout)
        mask = Variable(m, requires_grad=False) / (1 - self.dropout)
        mask = mask.expand_as(x)
        return mask * x

In [ ]:
class pBLSTM(torch.nn.Module):
    '''
    Pyramidal BiLSTM
    Read the write up/paper and understand the concepts and then write your implementation here.

    At each step,
    1. Pad your input if it is packed (Unpack it)
    2. Reduce the input length dimension by concatenating feature dimension
        (Tip: Write down the shapes and understand)
        (i) How should  you deal with odd/even length input?
        (ii) How should you deal with input length array (x_lens) after truncating the input?
    3. Pack your input
    4. Pass it into LSTM layer

    To make our implementation modular, we pass 1 layer at a time.
    '''
    def __init__(self, input_size, hidden_size):
        super(pBLSTM, self).__init__()

        self.blstm = torch.nn.LSTM(input_size *2 , hidden_size, num_layers=3, batch_first=True, bidirectional=True, dropout = 0.3)
        self._init_weights()

    def _init_weights(self):
        for name, param in self.blstm.named_parameters():
            if 'weight_ih' in name:
                torch.nn.init.xavier_uniform_(param.data)
            elif 'weight_hh' in name:
                torch.nn.init.orthogonal_(param.data)
            elif 'bias' in name:
                param.data.fill_(0)

    def forward(self, x_packed): # x_packed is a PackedSequence

        # TODO: Pad Packed Sequence
        x, x_lens = pad_packed_sequence(x_packed, batch_first=True)

        # Call self.trunc_reshape() which downsamples the time steps of x and increases the feature dimensions as mentioned above
        # self.trunc_reshape will return 2 outputs. What are they? Think about what quantites are changing.
        x, x_lens = self.trunc_reshape(x, x_lens)
        # TODO: Pack Padded Sequence. What output(s) would you get?
        x_packed = pack_padded_sequence(x, x_lens, batch_first=True, enforce_sorted=False)
        # TODO: Pass the sequence through bLSTM
        x, _ = self.blstm(x_packed)
        # What do you return?
        return x

    def trunc_reshape(self, x, x_lens):
        # TODO: If you have odd number of timesteps, how can you handle it? (Hint: You can exclude them)
        batch_size, seq_len, feat_dim = x.size()
        if seq_len % 2 == 1:
            x = x[:, :-1, :]
            x_lens = x_lens - 1
        # TODO: Reshape x. When reshaping x, you have to reduce number of timesteps by a downsampling factor while increasing number of features by the same factor
        x = x.reshape(batch_size, seq_len // 2, feat_dim * 2)
        # TODO: Reduce lengths by the same downsampling factor
        x_lens = x_lens // 2

        return x, x_lens

In [ ]:
class pBLSTMWithDropout(torch.nn.Module):
    '''
    Pyramidal BiLSTM
    Read the write up/paper and understand the concepts and then write your implementation here.

    At each step,
    1. Pad your input if it is packed (Unpack it)
    2. Reduce the input length dimension by concatenating feature dimension
        (Tip: Write down the shapes and understand)
        (i) How should  you deal with odd/even length input?
        (ii) How should you deal with input length array (x_lens) after truncating the input?
    3. Pack your input
    4. Pass it into LSTM layer

    To make our implementation modular, we pass 1 layer at a time.
    '''
    def __init__(self, input_size, hidden_size):
        super(pBLSTM, self).__init__()

        self.blstm = torch.nn.LSTM(input_size *2 , hidden_size, num_layers=3, batch_first=True, bidirectional=True, dropout = 0.3)
        self.locked_dropout = LockedDropout(0.3)
        self._init_weights()

    def _init_weights(self):
        for name, param in self.blstm.named_parameters():
            if 'weight_ih' in name:
                torch.nn.init.xavier_uniform_(param.data)
            elif 'weight_hh' in name:
                torch.nn.init.orthogonal_(param.data)
            elif 'bias' in name:
                param.data.fill_(0)

    def forward(self, x_packed,x_lens): # x_packed is a PackedSequence

        # TODO: Pad Packed Sequence
        x, x_lens = pad_packed_sequence(x_packed, batch_first=True)

        # Call self.trunc_reshape() which downsamples the time steps of x and increases the feature dimensions as mentioned above
        # self.trunc_reshape will return 2 outputs. What are they? Think about what quantites are changing.
        x, x_lens = self.trunc_reshape(x, x_lens)
        # TODO: Pack Padded Sequence. What output(s) would you get?
        x_packed = pack_padded_sequence(x, x_lens, batch_first=True, enforce_sorted=False)
        # TODO: Pass the sequence through bLSTM
        x, _ = self.blstm(x_packed)
        # What do you return?
        return x

    def trunc_reshape(self, x, x_lens):
        # TODO: If you have odd number of timesteps, how can you handle it? (Hint: You can exclude them)
        batch_size, seq_len, feat_dim = x.size()
        if seq_len % 2 == 1:
            x = x[:, :-1, :]
            x_lens = x_lens - 1
        # TODO: Reshape x. When reshaping x, you have to reduce number of timesteps by a downsampling factor while increasing number of features by the same factor
        x = x.reshape(batch_size, seq_len // 2, feat_dim * 2)
        # TODO: Reduce lengths by the same downsampling factor
        x_lens = x_lens // 2

        return x, x_lens

### Encoder

In [ ]:
# class pBLSTMWithDropout(torch.nn.Module):
#     def __init__(self, input_size, hidden_size, dropout=0.3):
#         super(pBLSTMWithDropout, self).__init__()
#         self.pBLSTM = torch.nn.LSTM(input_size, hidden_size, num_layers=1, bidirectional=True, batch_first=True)
#         self.dropout = LockedDropout(dropout=dropout)

#     def forward(self, x, x_lens):

#         # Pass through pBLSTM
#         packed_output, _ = self.pBLSTM(x)

#         # Unpack to apply dropout
#         output, _ = pad_packed_sequence(packed_output, batch_first=True)
#         output = self.dropout(output)

#         # Repack the output
#         packed_output = pack_padded_sequence(output, x_lens, batch_first=True, enforce_sorted=False)
#         return packed_output

In [ ]:
class Encoder(torch.nn.Module):
    '''
    The Encoder takes utterances as inputs and returns latent feature representations
    '''
    def __init__(self, input_size, encoder_hidden_size, num_pblstm_layers=3):
        super(Encoder, self).__init__()


        self.embedding = torch.nn.Sequential(
            nn.Conv1d(in_channels=input_size, out_channels=64, kernel_size=3, stride=1, padding=1),
            torch.nn.BatchNorm1d(64),
            torch.nn.GELU(),
            torch.nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1, stride=1),
            torch.nn.BatchNorm1d(128),
            torch.nn.GELU(),
            torch.nn.Conv1d(in_channels=128, out_channels=256, kernel_size=3, padding=1, stride=1),
            torch.nn.BatchNorm1d(256),
            torch.nn.GELU()
            # nn.Conv1d(in_channels=encoder_hidden_size//2, out_channels=encoder_hidden_size, kernel_size=3, stride=1, padding=1),
            # nn.ReLU(),
            # nn.BatchNorm1d(encoder_hidden_size),
        )
        # 256
        self._init_conv_weights()
        self.bLSTM = BiLSTM(encoder_hidden_size, encoder_hidden_size, num_layers=1)
        # pBLSTMs_list = [pBLSTM(encoder_hidden_size * (2**i) * 2, encoder_hidden_size * (2**i) * 2) for i in range(num_pblstm_layers)]

        self.pBLSTMs = torch.nn.Sequential( # How many pBLSTMs are required?
            # TODO: Fill this up with pBLSTMs - What should the input_size be?
            # Hint: You are downsampling timesteps by a factor of 2, upsampling features by a factor of 2 and the LSTM is bidirectional)

            # Optional: Dropout/Locked Dropout after each pBLSTM (Not needed for early submission)
            # https://github.com/salesforce/awd-lstm-lm/blob/dfd3cb0235d2caf2847a4d53e1cbd495b781b5d2/locked_dropout.py#L5
            # ...
            # ...
            # * pBLSTMs_list
            pBLSTM(encoder_hidden_size * 2, encoder_hidden_size),
            pBLSTM(encoder_hidden_size * 2, encoder_hidden_size),


        )
        self.locked_dropout = LockedDropout(0.3)

    def _init_conv_weights(self):
        for m in self.embedding.modules():
            if isinstance(m, nn.Conv1d):
                torch.nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    torch.nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                torch.nn.init.constant_(m.weight, 1)
                torch.nn.init.constant_(m.bias, 0)

    def forward(self, x, x_lens):
        # Where are x and x_lens coming from? The dataloader
        #TODO: Call the embedding layer
        x = PermuteBlock()(x)
        x = self.embedding(x)
        x = PermuteBlock()(x)
        # TODO: Pack Padded Sequence
        x_packed = pack_padded_sequence(x, x_lens, batch_first=True, enforce_sorted=False)
        # TODO: Pass Sequence through the pyramidal Bi-LSTM layer
        x = self.bLSTM(x_packed)
        for pBLSTM in self.pBLSTMs:
            x = pBLSTM(x)
            x, x_lens = pad_packed_sequence(x, batch_first=True)
            x = self.locked_dropout(x)
            x = pack_padded_sequence(x, x_lens, batch_first=True, enforce_sorted=False)
        # TODO: Pad Packed Sequence
        encoder_outputs, encoder_lens = pad_packed_sequence(x, batch_first=True)



        # Remember the number of output(s) each function returns

        return encoder_outputs, encoder_lens

### Decoder

In [ ]:
class Decoder(torch.nn.Module):

    def __init__(self, embed_size, output_size= 41, dropout=0.3):
        super().__init__()

        self.mlp = torch.nn.Sequential(
            PermuteBlock(), torch.nn.BatchNorm1d(embed_size), PermuteBlock(),
            #TODO define your MLP arch. Refer HW1P2
            #Use Permute Block before and after BatchNorm1d() to match the size
            torch.nn.Linear(embed_size, embed_size * 2),
            torch.nn.GELU(),
            torch.nn.Dropout(dropout),
            PermuteBlock(), torch.nn.BatchNorm1d(embed_size * 2), PermuteBlock(),
            torch.nn.Linear(embed_size * 2, embed_size // 2),
            torch.nn.GELU(),
            torch.nn.Dropout(dropout),
            PermuteBlock(), torch.nn.BatchNorm1d(embed_size // 2), PermuteBlock(),
            torch.nn.Linear(embed_size // 2, embed_size // 4),
            torch.nn.GELU(),
            torch.nn.Dropout(dropout),
            PermuteBlock(), torch.nn.BatchNorm1d(embed_size // 4), PermuteBlock(),
            # torch.nn.Linear(embed_size // 4, embed_size // 8),
            # torch.nn.ReLU(),
            # PermuteBlock(), torch.nn.BatchNorm1d(embed_size // 8), PermuteBlock(),
            torch.nn.Linear(embed_size // 4, output_size)
        )

        self.softmax = torch.nn.LogSoftmax(dim=2)
        self._init_weights()

    def _init_weights(self):
        for m in self.mlp.modules():
            if isinstance(m, nn.Linear):
                torch.nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    torch.nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                torch.nn.init.constant_(m.weight, 1)
                torch.nn.init.constant_(m.bias, 0)

    def forward(self, encoder_out):
        #TODO call your MLP
        #TODO Think what should be the final output of the decoder for the classification
        out = self.softmax(self.mlp(encoder_out))
        return out

In [ ]:
class ASRModel(torch.nn.Module):

    def __init__(self, input_size, embed_size= 256, output_size= len(PHONEMES)):
        super().__init__()

        self.augmentations  = torch.nn.Sequential(
            #TODO Add Time Masking/ Frequency Masking
            #Hint: See how to use PermuteBlock() function defined above
        )
        self.encoder        = Encoder(input_size, embed_size, num_pblstm_layers=3)
        self.decoder        = Decoder(512, output_size,dropout=0.3)



    def forward(self, x, lengths_x):

        if self.training:
            x = self.augmentations(x)

        encoder_out, encoder_lens   = self.encoder(x, lengths_x)
        decoder_out                 = self.decoder(encoder_out)

        return decoder_out, encoder_lens

## Initialize ASR Network

In [ ]:
model = ASRModel(
    input_size  = 28,
    embed_size  = 256,
    output_size = len(PHONEMES)
).to(device)
print(model)
summary(model, x.to(device), lx)

ASRModel(
  (augmentations): Sequential()
  (encoder): Encoder(
    (embedding): Sequential(
      (0): Conv1d(28, 64, kernel_size=(3,), stride=(1,), padding=(1,))
      (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): GELU(approximate='none')
      (3): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (4): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): GELU(approximate='none')
      (6): Conv1d(128, 256, kernel_size=(3,), stride=(1,), padding=(1,))
      (7): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (8): GELU(approximate='none')
    )
    (bLSTM): BiLSTM(
      (blstm): LSTM(256, 256, batch_first=True, bidirectional=True)
    )
    (pBLSTMs): Sequential(
      (0): pBLSTM(
        (blstm): LSTM(1024, 256, num_layers=3, batch_first=True, dropout=0.3, bidirectional=True)
      )
      (1): pBLSTM(
        (blstm): LSTM(1024, 256, num

,Kernel Shape,Output Shape,Params,Mult-Adds
Layer,,,,
0_augmentations,-,"[64, 1658, 28]",NaN,NaN
1_encoder.embedding.Conv1d_0,"[28, 64, 3]","[64, 64, 1658]",5440.0,8913408.0
2_encoder.embedding.BatchNorm1d_1,[64],"[64, 64, 1658]",128.0,64.0
3_encoder.embedding.GELU_2,-,"[64, 64, 1658]",NaN,NaN
4_encoder.embedding.Conv1d_3,"[64, 128, 3]","[64, 128, 1658]",24704.0,40747008.0
5_encoder.embedding.BatchNorm1d_4,[128],"[64, 128, 1658]",256.0,128.0
6_encoder.embedding.GELU_5,-,"[64, 128, 1658]",NaN,NaN
7_encoder.embedding.Conv1d_6,"[128, 256, 3]","[64, 256, 1658]",98560.0,162988032.0
8_encoder.embedding.BatchNorm1d_7,[256],"[64, 256, 1658]",512.0,256.0


# Training Config
Initialize Loss Criterion, Optimizer, CTC Beam Decoder, Scheduler, Scaler (Mixed-Precision), etc.

In [ ]:
#TODO


# Define CTC loss as the criterion. How would the losses be reduced?
# CTC Loss: https://pytorch.org/docs/stable/generated/torch.nn.CTCLoss.html
# Refer to the handout for hints
criterion = torch.nn.CTCLoss(blank=0, zero_infinity=True)

optimizer =  torch.optim.AdamW(model.parameters(), lr=config['lr'])

# Declare the decoder. Use the CTC Beam Decoder to decode phonemes
# CTC Beam Decoder Doc: https://github.com/parlance/ctcdecode
decoder = CTCBeamDecoder(labels=LABELS, model_path=None, beam_width=config['beam_width'], log_probs_input=True)

# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['epochs'], eta_min=0)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, threshold=0.02, verbose=True)

# Mixed Precision, if you need it
scaler = torch.cuda.amp.GradScaler()

# Decode Prediction

In [ ]:
def decode_prediction(output, output_lens, decoder, PHONEME_MAP= LABELS):

    # TODO: look at docs for CTC.decoder and find out what is returned here. Check the shape of output and expected shape in decode.
    beam_results, beam_scores, timesteps, out_lens = decoder.decode(output, seq_lens= output_lens) #lengths - list of lengths

    pred_strings                    = []

    for i in range(output_lens.shape[0]):
        #TODO: Create the prediction from the output of decoder.decode. Don't forget to map it using PHONEMES_MAP.
        # Get the sequence for this sample
        sequence = beam_results[i][0][:out_lens[i][0]] # Get first beam result up to its length

        # Map indices to phoneme labels
        pred_str = ' '.join([PHONEME_MAP[idx.item()] for idx in sequence])
        pred_strings.append(pred_str)

    return pred_strings

def calculate_levenshtein(output, label, output_lens, label_lens, decoder, PHONEME_MAP= LABELS): # y - sequence of integers

    dist            = 0
    batch_size      = label.shape[0]

    print(output.shape, label.shape)
    print(output_lens, label_lens)
    print((output_lens.shape[0]))
    pred_strings    = decode_prediction(output, output_lens, decoder, PHONEME_MAP)

    for i in range(batch_size):
        # TODO: Get predicted string and label string for each element in the batch
        pred_string = pred_strings[i]
        label_indices = label[i][:label_lens[i]]
        label_string = ' '.join([PHONEME_MAP[idx.item()] for idx in label_indices])
        dist += Levenshtein.distance(pred_string, label_string)

    dist /= batch_size # TODO: Uncomment this, but think about why we are doing this
    return dist

In [ ]:
def decode_prediction(output, output_lens, decoder, PHONEME_MAP=LABELS):
    # If shape is [T, B, C], transpose to [B, T, C]
    # if output.shape[1] == output_lens.shape[0]:  # Check if second dimension matches batch size

    ############## Use torch.permute instead of transpose #####################
    output = torch.permute(output, (1, 0, 2))

    beam_results, beam_scores, timesteps, out_lens = decoder.decode(output, seq_lens=output_lens)

    pred_strings = []
    batch_size = output.size(0)
    for i in range(batch_size):
        pred = beam_results[i][0][:out_lens[i][0]].numpy()
        # if i == 0:
        #     print(pred)
        pred_string = ''.join([PHONEME_MAP[idx] for idx in pred])
        # if i == 0:
        #     print(pred_string)
        pred_strings.append(pred_string)
    return pred_strings

# Calculate Levenshtein Distance Function
def calculate_levenshtein(output, label, output_lens, label_lens, decoder, PHONEME_MAP=LABELS):
    dist = 0
    batch_size = label.shape[0]

    # Ensure output is in the correct shape before passing to decode_prediction
    # if output.size(0) < output.size(1):  # If shape is [T, B, C]
    # output = output.transpose(0, 1)   # Change to [B, T, C]

    pred_strings = decode_prediction(output, output_lens, decoder, PHONEME_MAP)
    # print(batch_size)
    # print(len(pred_strings))
    # print(label.shape)
    # print(label_lens.shape)
    # print(output.shape)
    # print(output_lens.shape)
    for i in range(batch_size):
        pred_string = pred_strings[i]
        label_seq = label[i][:label_lens[i]].cpu().numpy()
        label_string = ''.join([PHONEME_MAP[idx] for idx in label_seq])
        dist += Levenshtein.distance(pred_string, label_string)

    dist /= batch_size  # Average over batch size
    return dist

# Test Implementation

In [ ]:
# test code to check shapes

model.eval()
for i, data in enumerate(val_loader, 0):
    x, y, lx, ly = data
    x, y = x.to(device), y.to(device)
    h, lh = model(x, lx)
    print(h.shape)
    h = torch.permute(h, (1, 0, 2))
    print(h.shape, y.shape)
    loss = criterion(h, y, lh, ly)
    print(loss)

    print(calculate_levenshtein(h, y, lx, ly, decoder, LABELS))

    break

torch.Size([64, 734, 41])
torch.Size([734, 64, 41]) torch.Size([64, 265])
tensor(7.6849, device='cuda:0', grad_fn=<MeanBackward0>)
208.765625


# WandB

You will need to fetch your api key from wandb.ai

In [ ]:
import wandb
wandb.login(key="0cc2199d5dbab5108d50abc33d06f093f13e6a18")

wandb: Using wandb-core as the SDK backend. Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: pkotchav (pkotchav-carnegie-mellon-university). Use `wandb login --relogin` to force relogin
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [ ]:
run = wandb.init(
    name = "final-submission-r4", ## Wandb creates random run names if you skip this field
    reinit = True, ### Allows reinitalizing runs when you re-run this cell
    # run_id = ### Insert specific run id here if you want to resume a previous run
    # resume = "must" ### You need this to resume previous runs, but comment out reinit = True when using this
    project = "hw3p2-ablations", ### Project should be created in your wandb account
    config = config ### Wandb Config for your run
)

# Train Functions

In [ ]:
from tqdm import tqdm

def train_model(model, train_loader, criterion, optimizer):

    model.train()
    batch_bar = tqdm(total=len(train_loader), dynamic_ncols=True, leave=False, position=0, desc='Train')

    total_loss = 0

    for i, data in enumerate(train_loader):
        optimizer.zero_grad()

        x, y, lx, ly = data
        x, y = x.to(device), y.to(device)

        with torch.cuda.amp.autocast():
            h, lh = model(x, lx)
            h = torch.permute(h, (1, 0, 2))
            # if i == 0:
                # print(h.shape)
            #     print(lh.shape)
            #     print(y.shape)
            #     print(ly.shape)
                # print(h)
            #     print(lh)
            #     print(y)
            #     print(ly)
            loss = criterion(h, y, lh, ly)

        total_loss += loss.item()

        batch_bar.set_postfix(
            loss="{:.04f}".format(float(total_loss / (i + 1))),
            lr="{:.06f}".format(float(optimizer.param_groups[0]['lr'])))

        batch_bar.update() # Update tqdm bar

        # Another couple things you need for FP16.
        scaler.scale(loss).backward() # This is a replacement for loss.backward()
        scaler.step(optimizer) # This is a replacement for optimizer.step()
        scaler.update() # This is something added just for FP16
        # loss.backward()
        # optimizer.step()

        del x, y, lx, ly, h, lh, loss
        torch.cuda.empty_cache()

    batch_bar.close() # You need this to close the tqdm bar

    return total_loss / len(train_loader)


def validate_model(model, val_loader, decoder, phoneme_map= LABELS):

    model.eval()
    batch_bar = tqdm(total=len(val_loader), dynamic_ncols=True, position=0, leave=False, desc='Val')

    total_loss = 0
    vdist = 0

    for i, data in enumerate(val_loader):

        x, y, lx, ly = data
        x, y = x.to(device), y.to(device)

        with torch.inference_mode():
            h, lh = model(x, lx)
            h = torch.permute(h, (1, 0, 2))
            loss = criterion(h, y, lh, ly)

        total_loss += float(loss)
        vdist += calculate_levenshtein(h, y, lh, ly, decoder, phoneme_map)

        batch_bar.set_postfix(loss="{:.04f}".format(float(total_loss / (i + 1))), dist="{:.04f}".format(float(vdist / (i + 1))))

        batch_bar.update()

        del x, y, lx, ly, h, lh, loss
        torch.cuda.empty_cache()

    batch_bar.close()
    total_loss = total_loss/len(val_loader)
    val_dist = vdist/len(val_loader)
    return total_loss, val_dist

## Training Setup

In [ ]:
def save_model(model, optimizer, scheduler, metric, epoch, path):
    torch.save(
        {'model_state_dict'         : model.state_dict(),
         'optimizer_state_dict'     : optimizer.state_dict(),
         'scheduler_state_dict'     : scheduler.state_dict(),
         metric[0]                  : metric[1],
         'epoch'                    : epoch},
         path
    )

def load_model(path, model, metric= 'valid_acc', optimizer= None, scheduler= None):

    checkpoint = torch.load(path)
    model.load_state_dict(checkpoint['model_state_dict'])

    if optimizer != None:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    if scheduler != None:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    epoch   = checkpoint['epoch']
    metric  = checkpoint[metric]

    return [model, optimizer, scheduler, epoch, metric]

In [ ]:
# This is for checkpointing, if you're doing it over multiple sessions

last_epoch_completed = 0
start = last_epoch_completed
end = config["epochs"]
best_lev_dist = float("inf") # if you're restarting from some checkpoint, use what you saw there.
epoch_model_path = '/content/hw3p2/finalbig/last.pth'#TODO set the model path( Optional, you can just store best one. Make sure to make the changes below )
best_model_path = '/content/hw3p2/finalbig/best.pth'#TODO set best model path

In [ ]:
torch.cuda.empty_cache()
gc.collect()

#TODO: Please complete the training loop

for epoch in range(0, config['epochs']):

    print("\nEpoch: {}/{}".format(epoch+1, config['epochs']))

    curr_lr = optimizer.param_groups[0]['lr'] #TODO

    train_loss              = train_model(model, train_loader, criterion, optimizer) #TODO
    valid_loss, valid_dist  = validate_model(model, val_loader, decoder, phoneme_map=LABELS) #TODO
    scheduler.step(valid_dist)

    print("\tTrain Loss {:.04f}\t Learning Rate {:.07f}".format(train_loss, curr_lr))
    print("\tVal Dist {:.04f}%\t Val Loss {:.04f}".format(valid_dist, valid_loss))


    wandb.log({
        'train_loss': train_loss,
        'valid_dist': valid_dist,
        'valid_loss': valid_loss,
        'lr'        : curr_lr
    })

    save_model(model, optimizer, scheduler, ['valid_dist', valid_dist], epoch, epoch_model_path)
    wandb.save(epoch_model_path)
    print("Saved epoch model")

    if valid_dist <= best_lev_dist:
        best_lev_dist = valid_dist
        save_model(model, optimizer, scheduler, ['valid_dist', valid_dist], epoch, best_model_path)
        wandb.save(best_model_path)
        print("Saved best model")
      # You may find it interesting to exlplore Wandb Artifcats to version your models
run.finish()


Epoch: 1/60


	Train Loss 3.4328	 Learning Rate 0.0020000
	Val Dist 66.6528%	 Val Loss 3.2015


wandb: WARNING Saving files without folders. If you want to preserve subdirectories pass base_path to wandb.save, i.e. wandb.save("/mnt/folder/file.h5", base_path="/mnt")


Saved epoch model
Saved best model

Epoch: 2/60


	Train Loss 2.0620	 Learning Rate 0.0020000
	Val Dist 23.1374%	 Val Loss 1.0749
Saved epoch model
Saved best model

Epoch: 3/60


...

	Train Loss 0.1144	 Learning Rate 0.0002500
	Val Dist 4.0351%	 Val Loss 0.2456
Saved epoch model

Epoch: 58/60


Epoch 00058: reducing learning rate of group 0 to 1.2500e-04.
	Train Loss 0.1128	 Learning Rate 0.0002500
	Val Dist 4.0192%	 Val Loss 0.2501
Saved epoch model

Epoch: 59/60


	Train Loss 0.1098	 Learning Rate 0.0001250
	Val Dist 3.9887%	 Val Loss 0.2451
Saved epoch model
Saved best model

Epoch: 60/60


	Train Loss 0.1079	 Learning Rate 0.0001250
	Val Dist 3.9582%	 Val Loss 0.2473
Saved epoch model
Saved best model


lr,█████████████████████████▄▄▄▄▂▂▂▂▂▁▁▁▁▁▁
train_loss,█▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
valid_dist,█▃▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
valid_loss,█▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
lr,0.00013
train_loss,0.1079
valid_dist,3.95821
valid_loss,0.24733


# Retrain

In [ ]:
root = '/content/11-785-f24-hw3p2/'

# Feel free to add more items here
config = {
    "beam_width" : 3,
    "lr"         : 2e-3,
    "epochs"     : 50,
    "batch_size" : 64  # Increase if your device can handle it
}

# You may pass this as a parameter to the dataset class above
# This will help modularize your implementation
transforms = [] # set of tranformations

In [ ]:
#TODO


# Define CTC loss as the criterion. How would the losses be reduced?
# CTC Loss: https://pytorch.org/docs/stable/generated/torch.nn.CTCLoss.html
# Refer to the handout for hints
criterion = torch.nn.CTCLoss(blank=0, zero_infinity=True)

optimizer =  torch.optim.AdamW(model.parameters(), lr=config['lr'])

# Declare the decoder. Use the CTC Beam Decoder to decode phonemes
# CTC Beam Decoder Doc: https://github.com/parlance/ctcdecode
decoder = CTCBeamDecoder(labels=LABELS, model_path=None, beam_width=config['beam_width'], log_probs_input=True)

# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['epochs'], eta_min=0)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, threshold=0.02, verbose=True)

# Mixed Precision, if you need it
scaler = torch.cuda.amp.GradScaler()

In [ ]:
model, optimizer, scheduler, epoch, metric = load_model("/content/hw3p2/final/last.pth", model,'valid_dist',optimizer,scheduler)

In [ ]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, threshold=0.02, verbose=True)

In [ ]:
run = wandb.init(
    name = "final-submission-contTrain", ## Wandb creates random run names if you skip this field
    reinit = True, ### Allows reinitalizing runs when you re-run this cell
    # run_id = ### Insert specific run id here if you want to resume a previous run
    # resume = "must" ### You need this to resume previous runs, but comment out reinit = True when using this
    project = "hw3p2-ablations", ### Project should be created in your wandb account
    config = config ### Wandb Config for your run
)

In [ ]:
last_epoch_completed = 0
start = last_epoch_completed
end = config["epochs"]
best_lev_dist = float("inf") # if you're restarting from some checkpoint, use what you saw there.
epoch_model_path = '/content/hw3p2/final/contTrain/last.pth'#TODO set the model path( Optional, you can just store best one. Make sure to make the changes below )
best_model_path = '/content/hw3p2/final/contTrain/best.pth'#TODO set best model path

In [ ]:
torch.cuda.empty_cache()
gc.collect()

#TODO: Please complete the training loop

for epoch in range(0, config['epochs']):

    print("\nEpoch: {}/{}".format(epoch+1, config['epochs']))

    curr_lr = optimizer.param_groups[0]['lr'] #TODO

    train_loss              = train_model(model, train_loader, criterion, optimizer) #TODO
    valid_loss, valid_dist  = validate_model(model, val_loader, decoder, phoneme_map=LABELS) #TODO
    scheduler.step(valid_dist)

    print("\tTrain Loss {:.04f}\t Learning Rate {:.07f}".format(train_loss, curr_lr))
    print("\tVal Dist {:.04f}%\t Val Loss {:.04f}".format(valid_dist, valid_loss))


    wandb.log({
        'train_loss': train_loss,
        'valid_dist': valid_dist,
        'valid_loss': valid_loss,
        'lr'        : curr_lr
    })

    save_model(model, optimizer, scheduler, ['valid_dist', valid_dist], epoch, epoch_model_path)
    wandb.save(epoch_model_path)
    print("Saved epoch model")

    if valid_dist <= best_lev_dist:
        best_lev_dist = valid_dist
        save_model(model, optimizer, scheduler, ['valid_dist', valid_dist], epoch, best_model_path)
        wandb.save(best_model_path)
        print("Saved best model")
      # You may find it interesting to exlplore Wandb Artifcats to version your models
run.finish()


Epoch: 1/50


	Train Loss 0.2502	 Learning Rate 0.0010000
	Val Dist 6.0536%	 Val Loss 0.2954
Saved epoch model
Saved best model

Epoch: 2/50


	Train Loss 0.2570	 Learning Rate 0.0010000
	Val Dist 5.9214%	 Val Loss 0.2851
Saved epoch model
Saved best model

Epoch: 3/50


	Train Loss 0.2586	 Learning Rate 0.0010000
	Val Dist 5.9716%	 Val Loss 0.2906
Saved epoch model

Epoch: 4/50


	Train Loss 0.2575	 Learning Rate 0.0010000
	Val Dist 5.9020%	 Val Loss 0.2913
Saved epoch model
Saved best model

Epoch: 5/50


...

	Train Loss 0.1612	 Learning Rate 0.0000039
	Val Dist 4.9707%	 Val Loss 0.2606
Saved epoch model

Epoch: 48/50


Epoch 00048: reducing learning rate of group 0 to 1.9531e-06.
	Train Loss 0.1612	 Learning Rate 0.0000039
	Val Dist 4.9675%	 Val Loss 0.2608
Saved epoch model
Saved best model

Epoch: 49/50


	Train Loss 0.1603	 Learning Rate 0.0000020
	Val Dist 4.9724%	 Val Loss 0.2608
Saved epoch model

Epoch: 50/50


	Train Loss 0.1606	 Learning Rate 0.0000020
	Val Dist 4.9773%	 Val Loss 0.2612
Saved epoch model


lr,█████████▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_loss,▇███▇█▇▇▆▅▄▄▄▄▄▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
valid_dist,█▇▇▇▇▇▆▆▆▄▄▃▃▄▃▂▂▂▁▁▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
valid_loss,█▆▇▇▇▆▆▅▃▃▃▃▄▃▃▂▂▂▂▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
lr,0.0
train_loss,0.16061
valid_dist,4.97735
valid_loss,0.26116


# Generate Predictions and Submit to Kaggle

In [ ]:
model2, _, _, _, _ = load_model("/content/hw3p2/final/contTrain/best.pth", model,'valid_dist')

In [ ]:
#TODO: Make predictions

# Follow the steps below:
# 1. Create a new object for CTCBeamDecoder with larger (why?) number of beams
# 2. Get prediction string by decoding the results of the beam decoder

TEST_BEAM_WIDTH = 10

test_decoder    = CTCBeamDecoder(labels=LABELS, model_path=None, beam_width=TEST_BEAM_WIDTH, log_probs_input=True)
results = []

model.eval()
print("Testing")
for data in tqdm(test_loader):

    x, lx   = data
    # print(x.shape)
    # print(lx.shape)
    x       = x.to(device)

    with torch.no_grad():
        h, lh = model(x, lx)
        h = torch.permute(h, (1, 0, 2))

    prediction_string= decode_prediction(h, lh, test_decoder, LABELS)

    results.extend(prediction_string)
    #TODO save the output in results array.

    del x, lx, h, lh
    torch.cuda.empty_cache()

Testing


100%|██████████| 41/41 [00:26<00:00,  1.56it/s]


In [ ]:
data_dir = f"random_submission.csv"
df = pd.read_csv(data_dir)
df.label = results
df.to_csv('submission5.csv', index = False)

In [ ]:
!kaggle competitions submit -c hw3p2-785-f24 -f submission.csv -m "I made it!"
